# ProcureSight AI — Supplier Analytics

## 1. Order Volume and Recorded Compliance

Compare each supplier's order count and recorded non-compliance rate.

Non-compliance rate = non-compliant orders / orders with known compliance.

Show counts alongside percentages to make the evidence behind each
rate visible. Differences are descriptive and may reflect order mix,
time period, or other factors rather than supplier performance alone.

The business definition of Compliance still requires clarification.

In [1]:
print("Python is ready")

Python is ready


In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_PATH = (
    PROJECT_ROOT / "data" / "processed"
    / "procurement_prepared.parquet"
)

# Load the prepared dataset independently of the other notebooks.
supplier_df = pd.read_parquet(DATA_PATH)

supplier_summary = (
    supplier_df.groupby("Supplier", dropna=False)
    .agg(
        Orders=("PO_ID", "size"),
        Known_Compliance=("Non_Compliant", "count"),
        Non_Compliant_Orders=(
            "Non_Compliant",
            lambda values: values.sum(min_count=1),
        ),
    )
    .reset_index()
)

# Use only orders with a known label in the denominator.
supplier_summary["Non_Compliance_Rate_%"] = (
    100
    * supplier_summary["Non_Compliant_Orders"]
    / supplier_summary["Known_Compliance"].where(
        supplier_summary["Known_Compliance"] > 0
    )
)

display(supplier_summary.round(2))

,Supplier,Orders,Known_Compliance,Non_Compliant_Orders,Non_Compliance_Rate_%
0,Alpha_Inc,141,141,9,6.38
1,Beta_Supplies,156,156,38,24.36
2,Delta_Logistics,171,171,67,39.18
3,Epsilon_Group,166,166,3,1.81
4,Gamma_Co,143,143,20,13.99


## 2. Supplier Order Volumes and Non-Compliance Rates

Compare order volumes alongside recorded non-compliance rates.

The dashed line represents the overall order-level non-compliance
rate, calculated across all known labels—not an average of supplier
percentages.

These comparisons describe the observed dataset. They do not establish
why differences exist or predict future supplier performance.

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Keep the same supplier order in both panels.
plot_df = supplier_summary.sort_values(
    "Non_Compliance_Rate_%",
    ascending=False,
).copy()

overall_rate = (
    100
    * supplier_summary["Non_Compliant_Orders"].sum()
    / supplier_summary["Known_Compliance"].sum()
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Purchase Order Volume",
        "Recorded Non-Compliance Rate",
    ),
    horizontal_spacing=0.15,
)

fig.add_trace(
    go.Bar(
        x=plot_df["Supplier"],
        y=plot_df["Orders"],
        text=plot_df["Orders"],
        textposition="outside",
        marker_color="#2563EB",
        hovertemplate="%{x}<br>Orders: %{y}<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=plot_df["Supplier"],
        y=plot_df["Non_Compliance_Rate_%"],
        text=plot_df["Non_Compliance_Rate_%"],
        texttemplate="%{text:.2f}%",
        textposition="outside",
        marker_color="#F97316",
        customdata=plot_df[
            ["Non_Compliant_Orders", "Known_Compliance"]
        ].to_numpy(),
        hovertemplate=(
            "%{x}<br>Non-compliance rate: %{y:.2f}%"
            "<br>Non-compliant orders: %{customdata[0]}"
            "<br>Orders with known labels: %{customdata[1]}"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.add_hline(
    y=overall_rate,
    line_dash="dash",
    line_color="#64748B",
    annotation_text=f"Overall: {overall_rate:.2f}%",
    annotation_position="top right",
    row=1,
    col=2,
)

fig.update_layout(
    title="Supplier Compliance: Counts and Rates",
    template="plotly_white",
    height=480,
    showlegend=False,
)

fig.update_xaxes(
    categoryorder="array",
    categoryarray=plot_df["Supplier"].tolist(),
    tickangle=-35,
)

fig.update_yaxes(
    title_text="Orders",
    range=[0, plot_df["Orders"].max() * 1.2],
    row=1,
    col=1,
)
fig.update_yaxes(
    title_text="Non-compliant orders (%)",
    ticksuffix="%",
    range=[0, 50],
    row=1,
    col=2,
)

fig.show()

### Initial Compliance Findings

- Delta_Logistics: 67 non-compliant orders out of 171 (39.18%).
- Beta_Supplies: 38 out of 156 (24.36%).
- Both exceed the overall recorded rate of 17.63%.
- Epsilon_Group: 3 out of 166 (1.81%), the lowest recorded rate.
- These observations do not establish the cause of the differences.

## 3. Item-Category Mix by Supplier

Compare the percentage of each supplier's orders in each item category.

Each bar totals 100%, allowing comparison despite different order
volumes. Hover details show the underlying counts.

Category mix provides context; it does not by itself explain
differences in compliance.

In [4]:
import plotly.express as px

# Count orders for each supplier-category combination.
category_counts = pd.crosstab(
    supplier_df["Supplier"],
    supplier_df["Item_Category"],
)

display(category_counts)

# Convert counts to percentages within each supplier.
category_share = category_counts.div(
    category_counts.sum(axis=1),
    axis=0,
) * 100

category_plot = (
    category_share
    .rename_axis(index="Supplier", columns="Item_Category")
    .reset_index()
    .melt(
        id_vars="Supplier",
        var_name="Item_Category",
        value_name="Share %",
    )
)

# Attach counts for the hover details.
count_plot = (
    category_counts
    .reset_index()
    .melt(
        id_vars="Supplier",
        var_name="Item_Category",
        value_name="Orders",
    )
)

category_plot = category_plot.merge(
    count_plot,
    on=["Supplier", "Item_Category"],
    validate="one_to_one",
)

fig = px.bar(
    category_plot,
    x="Supplier",
    y="Share %",
    color="Item_Category",
    barmode="stack",
    category_orders={
        "Supplier": supplier_summary.sort_values(
            "Non_Compliance_Rate_%",
            ascending=False,
        )["Supplier"].tolist()
    },
    hover_data={"Orders": True, "Share %": ":.1f"},
    title="Item-Category Mix Within Each Supplier",
    labels={"Item_Category": "Item category"},
    color_discrete_sequence=px.colors.qualitative.Safe,
    template="plotly_white",
)

fig.update_layout(
    height=480,
    xaxis_title=None,
    yaxis_title="Share of supplier orders (%)",
)
fig.update_yaxes(range=[0, 100], ticksuffix="%")
fig.update_xaxes(tickangle=-35)

fig.show()

Item_Category,Electronics,MRO,Office Supplies,Packaging,Raw Materials
Supplier,,,,,
Alpha_Inc,22,35,39,27,18
Beta_Supplies,32,26,41,30,27
Delta_Logistics,33,34,37,29,38
Epsilon_Group,36,39,34,34,23
Gamma_Co,29,30,23,28,33


### Category-Mix Findings

- Every supplier has orders in all five item categories.
- Supplier-category groups contain between 18 and 41 orders.
- Category shares differ, so overall supplier rates need context.
- Category mix alone does not explain the compliance differences.

## 4. Recorded Non-Compliance Within Each Category

Compare suppliers within the same item category.

Each heatmap cell shows the non-compliance rate and the number of
non-compliant orders divided by orders with known labels.

These groups are relatively small. A zero observed rate does not
mean zero future risk. Comparisons remain descriptive and do not
control for time, order size, or other differences.

In [5]:
import plotly.graph_objects as go

# Calculate counts and rates for each supplier-category group.
category_compliance = (
    supplier_df.groupby(
        ["Supplier", "Item_Category"],
        dropna=False,
    )
    .agg(
        Known_Labels=("Non_Compliant", "count"),
        Non_Compliant_Orders=(
            "Non_Compliant",
            lambda values: values.sum(min_count=1),
        ),
    )
    .reset_index()
)

category_compliance["Rate_%"] = (
    100 * category_compliance["Non_Compliant_Orders"]
    / category_compliance["Known_Labels"].where(
        category_compliance["Known_Labels"] > 0
    )
)

supplier_order = supplier_summary.sort_values(
    "Non_Compliance_Rate_%",
    ascending=False,
)["Supplier"].tolist()

rates = category_compliance.pivot(
    index="Supplier",
    columns="Item_Category",
    values="Rate_%",
).reindex(supplier_order)

# Include the numerator and denominator in every cell label.
category_compliance["Label"] = category_compliance.apply(
    lambda row: (
        f"{row['Rate_%']:.1f}%"
        f"<br>{int(row['Non_Compliant_Orders'])}"
        f"/{int(row['Known_Labels'])} orders"
        if row["Known_Labels"] > 0
        else "No known labels"
    ),
    axis=1,
)

labels = category_compliance.pivot(
    index="Supplier",
    columns="Item_Category",
    values="Label",
).reindex(index=rates.index, columns=rates.columns)

fig = go.Figure(
    go.Heatmap(
        z=rates.to_numpy(dtype=float),
        x=rates.columns.tolist(),
        y=rates.index.tolist(),
        text=labels.to_numpy(),
        texttemplate="%{text}",
        colorscale="Oranges",
        zmin=0,
        zmax=100,
        colorbar=dict(title="Rate (%)"),
        hovertemplate=(
            "%{y}<br>%{x}<br>%{text}<extra></extra>"
        ),
        hoverongaps=False,
    )
)

fig.update_layout(
    title="Recorded Non-Compliance by Supplier and Item Category",
    template="plotly_white",
    height=480,
    xaxis_title=None,
    yaxis_title=None,
)
fig.update_yaxes(autorange="reversed")

fig.show()

### Heatmap Interpretation

#### How to Read the Chart

- Rows represent suppliers; columns represent item categories.
- Each cell shows the recorded non-compliance percentage and its
  underlying count: non-compliant orders / orders with known labels.
- Darker orange indicates a higher rate; lighter shades indicate
  a lower rate.
- Colours represent percentages, not order volumes or financial loss.

#### Main Findings

1. **Delta_Logistics has the highest recorded rate in every category.**
   Its rates range from 31.6% in Raw Materials to 45.9% in Office
   Supplies. The pattern extends across categories rather than
   being confined to one type of purchase.

2. **Beta_Supplies has the second-highest rate in every category.**
   Its rates range from 21.9% in Electronics to 30.0% in Packaging.

3. **Epsilon_Group records low rates across all five categories.**
   It has no recorded non-compliance in Office Supplies or
   Packaging, with 34 orders in each group. These zero observations
   do not guarantee zero future risk.

4. **Some suppliers show variation between categories.**
   Gamma_Co records 3.4% in Electronics (1/29 orders), compared
   with 21.4% in Packaging (6/28 orders).

#### What This Adds to the Overall Comparison

Delta_Logistics remains highest and Beta_Supplies remains
second-highest when comparing orders within the same category.
Different category mixes alone therefore do not explain their
higher overall recorded rates.

However, grouping by category does not account for order timing,
order size, contract requirements, or differences in how compliance
was recorded. These results do not establish causation.

#### Limitations and Business Use

- Each supplier-category group contains only 18–41 orders.
  With group size held fixed, changing one order from compliant 
  to non-compliant changes the rate by approximately 2.4–5.6 percentage points
- The chart does not show statistical uncertainty.
- The meaning and reliability of the Compliance label still
  require clarification.
- Use these findings to prioritize record review, starting with
  Delta_Logistics across categories and Beta_Supplies.
- Do not use the chart alone to reject suppliers or predict
  future compliance.